### TF - IDF 방식 계산법

In [4]:
docs=[
    '영화가 너무 재미있었다.',
    '영화가 너무 지루하다.',
    '배우의 연기가 너무 좋았다.'
]

In [5]:
tokens=[doc.split() for doc in docs]
tokens

[['영화가', '너무', '재미있었다.'], ['영화가', '너무', '지루하다.'], ['배우의', '연기가', '너무', '좋았다.']]

In [6]:
#단어 사전을 생성 -> 1차원으로 데이터를 변경하고 중복 데이터를 재거
vocab1=[]

for token in tokens:
    for word in token:
        vocab1.append(word)

#리스트에서 중복 값을 제거 : 집합의 형태로 변경했다가 다시 리스트로 변환
vocab1=list(set(vocab1))
vocab1

['지루하다.', '배우의', '좋았다.', '재미있었다.', '연기가', '영화가', '너무']

In [7]:
vocab=list(set(sum(tokens,[])))
vocab

['지루하다.', '배우의', '좋았다.', '재미있었다.', '연기가', '영화가', '너무']

In [8]:
#TF, IDF 계산
import math

In [9]:
#단어 사전의 길이
V=len(vocab)
#전체 문서의 길이
N=len(docs)

In [10]:
#단어의 개수 생성
word_cnt={
    w : sum(1 for doc in tokens if w in doc) for w in vocab
}
word_cnt

{'지루하다.': 1, '배우의': 1, '좋았다.': 1, '재미있었다.': 1, '연기가': 1, '영화가': 2, '너무': 3}

In [58]:
#TF 계산식 함수
def tf(word, doc):
    #word : 단어 사전의 각 원소를 대입 
    #doc : tokens의 각 원소를 대입
    result = math.log(doc.count(word)+1)
    #doc.count(word) : 문장에서 특정 단어의 개수
    #len(doc) : 문장의 단어의 개수
    return result

In [59]:
#IDF 계산 함수
#분모에 1을 더한 이유는 분모가 0이 되는 것을 방지하기 위함.
#분자에 1을 더한 이유는 (가상의 문서가 하나 더 존재한다) 결과에서 1 더해준다.
def idf(word):
    #word : 단어 사전의 각 원소를 대입
    result=math.log((N)+1 / (word_cnt[word]+1))+1
    #N : docs의 길이 -> 문장들의 개수
    #word_cnt[word] : 전체 문서에서 특정 단어의 개수
    return result

In [60]:
X_tfidf = [
    [  tf(w, doc) * idf(w) for w in vocab]  for doc in tokens
]
X_tfidf

[[0.0,
  0.0,
  0.0,
  1.5614963000824171,
  0.0,
  1.5276775353493184,
  1.5101285681270502],
 [1.5614963000824171,
  0.0,
  0.0,
  0.0,
  0.0,
  1.5276775353493184,
  1.5101285681270502],
 [0.0,
  1.5614963000824171,
  1.5614963000824171,
  0.0,
  1.5614963000824171,
  0.0,
  1.5101285681270502]]

In [61]:
import pandas as pd

In [62]:
pd.DataFrame(X_tfidf, columns=vocab)

,지루하다.,배우의,좋았다.,재미있었다.,연기가,영화가,너무
0,0.000000,0.000000,0.000000,1.561496,0.000000,1.527678,1.510129
1,1.561496,0.000000,0.000000,0.000000,0.000000,1.527678,1.510129
2,0.000000,1.561496,1.561496,0.000000,1.561496,0.000000,1.510129


In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [64]:
vec=TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=1
)

In [65]:
X=vec.fit_transform(docs)

In [66]:
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())

,뛰어났다,배우의,별로다,별로라서,보기,어렵다,연기가,연기였다,영화,영화가,영화는,재미있었다,정말,지루한,지루했다,훌륭한
0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.611713,0.0,0.611713,0.501613,0.0,0.000000,0.000000
1,0.000000,0.000000,0.707107,0.000000,0.0,0.0,0.000000,0.000000,0.707107,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
2,0.611713,0.611713,0.000000,0.000000,0.0,0.0,0.501613,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.5,0.5,0.000000,0.000000,0.000000,0.000000,0.5,0.000000,0.000000,0.5,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.611713,0.000000,0.000000,0.0,0.000000,0.501613,0.0,0.000000,0.611713
5,0.000000,0.000000,0.000000,0.611713,0.0,0.0,0.501613,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.611713,0.000000


### LSA(잠재 의미 분석)
- 문서 안에서 단어 사이의 잠재적인 의미 구조를 추출하는 기법
- TF-IDF 방식은 단어 간의 의미적 유사성 반영 X
- TF_IDF 방식에서 SVD(특이값 분해)를 하여 단어 간의 의미를 파악
- 차원 축소를 통해서 관계성을 확인
- LSA 효과
    - 벡터 공간의 차원을 줄여서 계산 효율의 증가 (차원 축소)
    - '영화','필름' 등 비슷한 문맥의 단어를 가까운 벡터로 이동 (의미 유추)
    - 문서들을 주제별로 분류 (토픽 분석)
- TruncatedSVD (차원 축소 모델)
    - 절단된 특이값 분해
    - 고차원 희소행렬(값이 0인 행렬)을 낮은 차원으로 압축하여 데이터의 구조적 의미를 유지
    - 자연어 처리, 추천 시스템, 의미 분석, 잠재적인 토픽 분석에서 주로 사용
    - TF-IDF 행렬은 고차원 -> 저차원
    - 0으로 이루어진 희소행렬들을 구조적인 의미를 유지하면서 값들을 부여
    - 같은 토픽의 문서는 같은 벡터 공간에서 가깝게 위치 -> 유사도 기반 자연어 처리에서 활용

In [67]:
from sklearn.decomposition import TruncatedSVD
from konlpy.tag import Okt

In [68]:
okt = Okt()
def tokenize(text):
    result = []
    for word, pos in okt.pos(text):
        if pos in ['Noun', 'Adjective', 'Verb']:
            result.append(word)
    return result

In [69]:
docs=[
    '이 영화가 정말 재미있었다.',
    '이 영화 별로다.',
    '배우의 연기가 뛰어났다.',
    '지루한 영화는 보기 어렵다',
    '정말 훌륭한 연기였다',
    '연기가 별로라서 지루했다'
]

In [70]:
tfidf = TfidfVectorizer(
    tokenizer=tokenize, 
    ngram_range= (1,1), 
    min_df=1, 
    max_df=0.8, 
    lowercase=False
)

In [24]:
X_tfidf=tfidf.fit_transform(docs)

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [25]:
X_tfidf.shape

(6, 14)

In [26]:
#차원 축소
lsa=TruncatedSVD(n_components=2, random_state=42)
X_lsa=lsa.fit_transform(X_tfidf)

In [27]:
df_lsa=pd.DataFrame(X_lsa, columns=['topic1','topic2'])
df_lsa

,topic1,topic2
0,0.722615,-0.336336
1,0.801574,-0.279071
2,0.229990,0.678032
3,0.346712,-0.383244
4,0.392081,0.481308
5,0.516723,0.493418


In [28]:
df_lsa['doucment']=docs
df_lsa

,topic1,topic2,doucment
0,0.722615,-0.336336,이 영화가 정말 재미있었다.
1,0.801574,-0.279071,이 영화 별로다.
2,0.229990,0.678032,배우의 연기가 뛰어났다.
3,0.346712,-0.383244,지루한 영화는 보기 어렵다
4,0.392081,0.481308,정말 훌륭한 연기였다
5,0.516723,0.493418,연기가 별로라서 지루했다


In [29]:
features=tfidf.get_feature_names_out()
features

array(['뛰어났다', '배우', '별로', '보기', '어렵다', '연기', '였다', '영화', '이', '재미있었다',
       '정말', '지루한', '지루했다', '훌륭한'], dtype=object)

In [77]:
components=lsa.components_
components

array([[ 2.78580010e-02,  3.50174274e-02,  1.34691193e-03, ...,
         2.27806351e-03,  1.79250990e-03,  9.76074462e-04],
       [-2.89788616e-02, -2.83608092e-02, -2.59746733e-03, ...,
        -2.92130616e-03,  5.55172509e-04, -3.70315576e-03],
       [-1.79101771e-02, -4.70370496e-02, -1.76423643e-03, ...,
        -4.55112680e-03, -2.90204067e-04,  1.33176455e-02],
       ...,
       [-2.70648135e-02, -2.03094338e-02,  8.93329465e-04, ...,
        -3.33561364e-04, -7.18428897e-03, -7.26483112e-03],
       [ 1.04393047e-02, -8.73767705e-03,  3.90651787e-03, ...,
        -4.53754144e-03,  5.00904557e-03,  2.59149477e-03],
       [ 2.65958640e-02, -8.49798229e-05,  5.88085281e-03, ...,
        -3.14770122e-03, -5.33492881e-03,  2.34336254e-02]],
      shape=(200, 4730))

In [78]:
components.shape

(200, 4730)

In [32]:
pd.DataFrame(components, index=['topic1','topic2'], columns=features).T

,topic1,topic2
뛰어났다,0.083061,0.338339
배우,0.083061,0.338339
별로,0.441011,0.083596
보기,0.105700,-0.161434
어렵다,0.105700,-0.161434
연기,0.283132,0.564684
였다,0.125589,0.213017
영화,0.476112,-0.333026
이,0.477259,-0.262077
재미있었다,0.244520,-0.157252


1. ratings_train.txt 파일을 로드
2. 결측치 제외, id 컬럼 제외
3. document의 중복 데이터 제거
4. 상위 5000개의 데이터를 추출
5. 독립 변수, 종속 변수 분할, train_test_split으로 분할
6. X_train을 이용하여 tfidf, lsa 작업
7. SVC 모델을 이용하여 성능 평가

In [33]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [34]:
df=pd.read_csv('../../data/ratings_train.txt', sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [35]:
df.dropna(inplace=True)
df.drop('id',axis=1, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  149995 non-null  object
 1   label     149995 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.4+ MB


In [36]:
df.drop_duplicates('document',inplace=True)

In [37]:
df2=df.iloc[:5000]
df2['label'].value_counts()

label
0    2502
1    2498
Name: count, dtype: int64

In [38]:
X=df2['document'].values
Y=df2['label'].values

X_train,X_test,y_train,y_test=train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

In [39]:
okt=Okt()

def tokenize(text):
    return okt.morphs(text)

vec=TfidfVectorizer(
    tokenizer=tokenize,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=3
)

In [40]:
X_train_vec=vec.fit_transform(X_train)
X_test_vec=vec.transform(X_test)

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [41]:
svc=SVC(
    kernel='linear',
    C=1.0,
    random_state=42
)

In [42]:
X_train_vec.shape

(4000, 4730)

In [43]:
svc.fit(X_train_vec,y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [44]:
pred_tfidf=svc.predict(X_test_vec)

print(classification_report(pred_tfidf, y_test))

              precision    recall  f1-score   support

           0       0.79      0.75      0.77       528
           1       0.73      0.78      0.75       472

    accuracy                           0.76      1000
   macro avg       0.76      0.76      0.76      1000
weighted avg       0.76      0.76      0.76      1000



In [45]:
#SVD를 이용하여 차원 측소 -> 200개의 피쳐로 축소
lsa=TruncatedSVD(
    n_components=200,
    random_state=42
)

In [46]:
X_train_lsa=lsa.fit_transform(X_train_vec)
X_test_lsa=lsa.fit_transform(X_test_vec)

X_train_lsa.shape

(4000, 200)

In [47]:
svc.fit(X_train_lsa, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [48]:
pred_lsa=svc.predict(X_test_lsa)

print(classification_report(pred_lsa, y_test))

              precision    recall  f1-score   support

           0       0.55      0.52      0.53       524
           1       0.50      0.52      0.51       476

    accuracy                           0.52      1000
   macro avg       0.52      0.52      0.52      1000
weighted avg       0.52      0.52      0.52      1000



In [49]:
pd.DataFrame(X_train_lsa).describe()

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,...,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,0.118685,-0.010759,0.004849,0.002009,-0.003976,-0.001008,0.007311,0.001171,0.002404,0.004920,...,0.000210,-0.000568,-0.000089,0.000481,0.000017,0.000216,-0.000199,-0.000172,-0.000174,0.000045
std,0.068499,0.086031,0.078161,0.076490,0.073194,0.069785,0.067781,0.065841,0.064559,0.063203,...,0.029597,0.029519,0.029461,0.029447,0.029343,0.029288,0.029129,0.029051,0.028998,0.028893
min,0.000000,-0.631690,-0.600462,-0.250477,-0.438434,-0.427388,-0.360405,-0.327136,-0.498296,-0.264003,...,-0.145058,-0.139522,-0.123543,-0.147376,-0.129396,-0.115767,-0.167819,-0.141596,-0.131841,-0.122814
25%,0.068718,-0.038595,-0.018661,-0.041109,-0.033871,-0.034369,-0.022139,-0.035314,-0.023258,-0.030577,...,-0.017771,-0.017090,-0.016204,-0.016347,-0.016689,-0.017290,-0.016933,-0.016610,-0.016381,-0.016581
50%,0.112303,-0.012983,0.004720,-0.006802,-0.004524,-0.006823,0.005304,0.000267,0.005771,0.000348,...,-0.000541,-0.000774,0.000000,0.000000,0.000000,-0.000146,0.000000,-0.000376,0.000300,-0.000161
75%,0.160798,0.014901,0.021254,0.026179,0.030148,0.020211,0.031642,0.033742,0.025662,0.028730,...,0.016328,0.016811,0.017238,0.016204,0.016469,0.016081,0.015797,0.015983,0.016445,0.016032
max,0.437618,0.637658,0.709924,0.650996,0.500565,0.666413,0.639381,0.413127,0.691132,0.381986,...,0.158448,0.128762,0.155009,0.193957,0.155508,0.207477,0.137329,0.130945,0.208474,0.172008


In [50]:
from sklearn.preprocessing import StandardScaler

In [51]:
std=StandardScaler()

In [52]:
X_train_sc=std.fit_transform(X_train_lsa)
X_test_sc=std.fit_transform(X_test_lsa)

In [53]:
svc.fit(X_train_sc,y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [54]:
pred_sc=svc.predict(X_test_sc)

print(classification_report(pred_sc, y_test))

              precision    recall  f1-score   support

           0       0.57      0.54      0.55       526
           1       0.52      0.54      0.53       474

    accuracy                           0.54      1000
   macro avg       0.54      0.54      0.54      1000
weighted avg       0.54      0.54      0.54      1000



In [55]:
#TF-IDF에서 MaxAbsScaler를 사용하고 모델 성능 평가
from sklearn.preprocessing import MaxAbsScaler

In [56]:
ma=MaxAbsScaler()

X_train_ma=ma.fit_transform(X_train_vec)
X_test_ma=ma.transform(X_test_vec)

svc.fit(X_train_ma, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [57]:
pred_ma=svc.predict(X_test_ma)

print(classification_report(pred_ma, y_test))

              precision    recall  f1-score   support

           0       0.75      0.73      0.74       509
           1       0.73      0.74      0.73       491

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000

